# 03 - Feature engineering

Purpose: build the supervised frame and confirm it contains no future information.

## Leakage contract

A row predicts rainfall accumulated over `[target_date, target_date + 1)` for a
district, issued on `issue_date`. Every feature is derived from observations strictly
before `issue_date`:

* rolling windows are computed on `shift(1)` series, so the issue day itself is not
  an input;
* the day-of-year term uses the target date, which is known when the forecast is
  issued and is therefore not leakage;
* the regional term is a same-day cross-sectional mean of already-computed features.

`assert_no_leakage` fails the run if the antecedent term starts mirroring the target.

In [ ]:
import sys
from pathlib import Path

# The notebook kernel runs with the notebook directory as the working directory,
# but paths are resolved from the repository root so cells behave the same when
# executed by nbconvert, JupyterLab or an IDE.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "ml" / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

ML_SRC = REPO_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = REPO_ROOT / "data" / "processed"
ARTIFACT_ROOT = REPO_ROOT / "ml" / "artifacts"
CONFIG_PATH = REPO_ROOT / "ml" / "configs" / "training.yaml"
OBSERVATIONS = DATA_DIR / "observations.csv"
DISTRICTS = DATA_DIR / "districts.csv"

print("notebook environment ready")
print("repo root:", REPO_ROOT)
print("python", sys.version.split()[0])


In [ ]:
from rainml.data.loaders import load_districts, load_observations, attach_coordinates
from rainml.data.cleaning import clean_observations
from rainml.features.engineering import (
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    assert_no_leakage,
    build_training_frame,
)
from rainml.features.selection import analyse_features
from rainml.data.validation import validate_training_frame, chronological_split, validate_split

cleaned, _ = clean_observations(
    attach_coordinates(load_observations(OBSERVATIONS), load_districts(DISTRICTS))
)
frame = build_training_frame(cleaned, horizon_days=1)
print(len(frame), "supervised rows")
assert_no_leakage(frame)
print("leakage check passed")

## Feature definitions

| Feature | Meaning |
| --- | --- |
| `antecedent_rain_24h_mm` | rainfall on the day before issue |
| `antecedent_rain_72h_mm` | rainfall over the three days before issue |
| `rain_last_Nd_mm` | accumulated rainfall over the N days before issue |
| `dry_streak_days` | consecutive dry days before issue |
| `recent_event_count_30d` | heavy-rain days in the previous 30 days |
| `recent_mean/std/max_daily_mm_30d` | rainfall intensity statistics over 30 days |
| `doy_sin`, `doy_cos` | seasonal phase of the target date |
| `lat` | centroid latitude |
| `regional_mean_rain_7d_mm` | same-day cross-sectional 7-day mean |

In [ ]:
frame[["district_id", "issue_date", "target_date", *FEATURE_COLUMNS, TARGET_COLUMN]].head(8).round(3)

## Explicit leakage test

If a feature had been built from the target day, the antecedent column would equal
the target far more often than chance allows.

In [ ]:
check = (
    frame.assign(equal=frame["antecedent_rain_24h_mm"] == frame[TARGET_COLUMN])
    .groupby("district_id")["equal"]
    .mean()
)
check.describe()

## Correlation with the target

Ranking by absolute correlation is a first pass; the selection step uses f_regression
because the rainfall target is heavily skewed.

In [ ]:
correlations = frame[list(FEATURE_COLUMNS)].corrwith(frame[TARGET_COLUMN]).sort_values(key=abs, ascending=False)
correlations.round(4).to_frame("pearson_r")

## Feature selection

Selection is done on the full frame here for inspection. The training pipeline refits
the selector inside the pipeline so the choice cannot leak across the split.

In [ ]:
selection = analyse_features(frame, max_features=14)
print("selected:", selection.selected)
print("dropped :", selection.dropped)
print("constant:", selection.constant)
pd.Series(selection.scores).sort_values(ascending=False).round(2).to_frame("f_score")

## Chronological split

Random splitting is invalid here. Daily rainfall is strongly autocorrelated, so a random
split places near-duplicates of a test day in the training set and inflates every
score. The split is by date: train, then validation, then the most recent test slice.

In [ ]:
train, valid, test = chronological_split(frame, valid_fraction=0.2, test_fraction=0.2)
summary = pd.DataFrame(
    {
        "rows": [len(train), len(valid), len(test)],
        "first_issue_date": [train["issue_date"].min(), valid["issue_date"].min(), test["issue_date"].min()],
        "last_issue_date": [train["issue_date"].max(), valid["issue_date"].max(), test["issue_date"].max()],
    },
    index=["train", "valid", "test"],
)
summary

In [ ]:
split_report = validate_split(train, valid, test)
print("split ok:", split_report.ok, split_report.errors)
frame_report = validate_training_frame(frame)
print("frame ok:", frame_report.ok, frame_report.errors, frame_report.warnings)

## Target distribution by split

A large shift between splits means the test slice is not comparable to training, and
the scores must be read with that in mind.

In [ ]:
pd.DataFrame(
    {
        "split": ["train", "valid", "test"],
        "mean_target_mm": [train[TARGET_COLUMN].mean(), valid[TARGET_COLUMN].mean(), test[TARGET_COLUMN].mean()],
        "heavy_rate": [
            (train[TARGET_COLUMN] >= 15).mean(),
            (valid[TARGET_COLUMN] >= 15).mean(),
            (test[TARGET_COLUMN] >= 15).mean(),
        ],
    }
).round(4)